# NHS GP Cloud Based Telephony · **Bronze layer**

**What this notebook does, in one breath:** NHS England Digital publishes a monthly count
of every phone call into GP practices that use cloud-based telephony — practice by
practice, day by day. This notebook checks that every raw file I downloaded is still
exactly what NHS England published, saves what each publication page said on the day,
and copies the CSVs into Parquet **without changing a single value**.

**Rule for this layer:** nothing is cleaned, renamed, typed or dropped. Every value is
stored as the text that was in the file. Cleaning happens in Silver, where every
decision can be seen and argued with.

The download itself is `src/pipeline/download_bronze.py`, which writes the raw files and
a SHA-256 manifest (`evidence/download_manifest_<date>.txt`). This notebook starts from
that manifest.

## Step 0 — Tools I need

`pyarrow` rather than pandas for the CSVs: one month of the day-and-time file is about
4 million rows, and reading it as text through pandas costs gigabytes of memory.
`pyarrow` reads it as a column store, keeps every value as a string, and writes it
straight to Parquet. `BeautifulSoup` pulls the caveat text out of the page snapshots.

In [1]:
import hashlib
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import requests
from bs4 import BeautifulSoup

print("Tools loaded OK")

Tools loaded OK


## Step 1 — Point at my folders

Same project-root trick as every notebook: if Jupyter's current folder is `notebooks/`,
step up one level so the paths work from the notebook or from a terminal.

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
EVIDENCE_DIR = PROJECT_DIR / "evidence"
PAGES_DIR = EVIDENCE_DIR / "publication_pages"
print("Project :", PROJECT_DIR)
print("Bronze  :", BRONZE_DIR)
print("Evidence:", EVIDENCE_DIR)

Project : /Users/yusufismail/Documents/nhs-gp-cloud-telephony-pipeline
Bronze  : /Users/yusufismail/Documents/nhs-gp-cloud-telephony-pipeline/data/bronze
Evidence: /Users/yusufismail/Documents/nhs-gp-cloud-telephony-pipeline/evidence


## Step 2 — Is every file still what NHS England published?

The manifest records a SHA-256 fingerprint for every file at the moment it was
downloaded. Here I recompute each fingerprint and compare.

- **Present and matching** — fine.
- **Missing** — expected for the October 2025 – July 2026 back series, which isn't
  committed to git (about 330 MB). I re-download it from the URL in the manifest and
  check the new file against the old fingerprint.
- **Fingerprint differs** — either the file was edited locally, or NHS England reissued
  it under the same URL. Either way it's a stop: I report it and don't overwrite.

The most recent manifest wins if there's more than one.

In [3]:
def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_manifest() -> tuple[Path, pd.DataFrame]:
    path = sorted(EVIDENCE_DIR.glob("download_manifest_*.txt"))[-1]
    return path, pd.read_csv(path, sep="\t", dtype=str)


def verify_manifest(manifest: pd.DataFrame) -> pd.DataFrame:
    results = []
    for row in manifest.itertuples():
        path = PROJECT_DIR / row.path
        status = "ok"
        if not path.exists():
            r = requests.get(row.source_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=300)
            r.raise_for_status()
            if hashlib.sha256(r.content).hexdigest() != row.sha256:
                status = "MISMATCH on re-download (source reissued?) - not saved"
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(r.content)
                status = "re-downloaded, matches"
        elif sha256_of(path) != row.sha256:
            status = "MISMATCH on disk"
        results.append({"path": row.path, "status": status})
    return pd.DataFrame(results)


MANIFEST_PATH, manifest = load_manifest()
check = verify_manifest(manifest)
print(f"Manifest: {MANIFEST_PATH.name} - {len(manifest)} files")
print(check["status"].value_counts().to_string())
bad = check[check["status"].str.startswith("MISMATCH")]
assert bad.empty, f"Stop: fingerprints don't match the manifest:\n{bad.to_string()}"

Manifest: download_manifest_2026-10-02.txt - 85 files
status
ok    85


## Step 3 — What did each publication page say on the day?

The coverage caveat moves month to month, and some of my argument rests on exactly what
the page said. For each page snapshot I pull out the title, the "Summary" and "Key Facts"
sections and the "Last edited" stamp. These go into a plain-text file beside the HTML,
headed with the source URL, retrieval time and SHA-256 from the manifest. Both files are
committed.

One thing worth keeping in mind from the August 2026 Key Facts: the page defines
**"dealt with"** as answered + call back requested + ended in the IVR (24,344,425, 89.6%),
and **"not dealt with"** as calls that ended while waiting in the queue (2,817,449, 10.4%).
That second group is what the files label "Missed".

In [4]:
def extract_caveat_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    main = soup.find("main") or soup
    lines, keep = [], False
    for tag in main.find_all(["h1", "h2", "p", "li"]):
        text = tag.get_text(" ", strip=True)
        if not text:
            continue
        if tag.name == "h1":
            lines.append(f"# {text}")
        elif tag.name == "h2":
            keep = text in ("Summary", "Key Facts")
            if keep:
                lines.append(f"\n## {text}")
        elif keep and tag.name == "p":
            lines.append(text)
        if text.startswith("Last edited"):
            lines.append(f"\n{text}")
    return "\n".join(lines) + "\n"


def write_caveats(manifest: pd.DataFrame) -> pd.DataFrame:
    pages = manifest[manifest["path"].str.startswith("evidence/publication_pages/")]
    written = []
    for row in pages.itertuples():
        html_path = PROJECT_DIR / row.path
        out = html_path.with_name(html_path.stem.replace("page_", "caveats_") + ".txt")
        header = (f"Source URL : {row.source_url}\n"
                  f"Retrieved  : {row.downloaded_utc}\n"
                  f"HTML SHA256: {row.sha256}\n\n")
        out.write_text(header + extract_caveat_text(html_path.read_text(encoding="utf-8")))
        written.append({"folder": html_path.parent.name, "file": out.name})
    return pd.DataFrame(written)


caveats = write_caveats(manifest)
print(f"Wrote {len(caveats)} caveat files")
print((PAGES_DIR / "2026-08" / caveats.loc[caveats.folder == "2026-08", "file"].iloc[0]).read_text()[:1500])

Wrote 13 caveat files
Source URL : https://digital.nhs.uk/data-and-information/publications/statistical/cloud-based-telephony-data-in-general-practice/august-2026
Retrieved  : 2026-10-02T14:38:21+00:00
HTML SHA256: d5c89df42ad6c24ef0a797c9850bc527310e62af2fef4c004dc99c6a82231ca0

# Cloud Based Telephony Data in General Practice, August 2026

## Summary
This publication provides information about the number of cloud based telephony calls received by general practices. This information is published monthly and includes practice-level data.
The publication presents a high-level time series from February 2025 onwards, showing monthly counts grouped by day and time of call of:
• The total count of practices included in this publication
• The total number of inbound calls into general practice
The outcome of those calls grouped into:
• Calls dealt with by either being answered, ending during the Interactive Voice Response (IVR) stage of the call prior to joining the queue to be answered, or 

## Step 4 — Telephony CSVs into Parquet, unchanged

Each month has up to three ZIPs, and each ZIP holds eight CSVs: one per NHS region, plus
"Unmapped" for calls NHS England couldn't attach to a practice. I combine the eight into
one Parquet file per ZIP and add four lineage columns, each starting with `_` so they
can't be confused with source columns:

| Column | Meaning |
|---|---|
| `_publication_month` | the month folder, e.g. `2026-08` |
| `_source_file` | the ZIP it came from |
| `_source_member` | the CSV inside the ZIP |
| `_source_sha256` | the ZIP's fingerprint, matching the manifest |

Every source column is read as a string, and empty cells stay as empty strings rather
than becoming nulls. So `Value` stays `"1.0"`, `PRACTICE_CODE` keeps `"Unassigned"`, and
nothing is guessed.

The row-count check counts lines in the raw CSV and compares them with the rows written.
If any line was dropped or split, this fails.

In [5]:
KINDS = {
    "By Day and Time": "day_time",
    "By Durations": "durations",
    "Calls Answered Metric": "calls_answered",
}


def read_csv_as_text(raw: bytes) -> pa.Table:
    header = raw.split(b"\n", 1)[0].decode("utf-8-sig").strip().split(",")
    return pacsv.read_csv(
        pa.BufferReader(raw),
        read_options=pacsv.ReadOptions(encoding="utf-8"),
        convert_options=pacsv.ConvertOptions(
            column_types={c: pa.string() for c in header},
            strings_can_be_null=False,
        ),
    )


def count_data_lines(raw: bytes) -> int:
    return raw.count(b"\n") - 1 + (0 if raw.endswith(b"\n") else 1)


def with_lineage(table: pa.Table, **lineage: str) -> pa.Table:
    for name, value in lineage.items():
        table = table.append_column(name, pa.array([value] * table.num_rows, pa.string()).dictionary_encode())
    return table


def zip_to_parquet(zip_path: Path, out_path: Path, month: str, sha: str) -> dict:
    tables, raw_rows = [], 0
    with zipfile.ZipFile(zip_path) as zf:
        for member in sorted(zf.namelist()):
            raw = zf.read(member)
            raw_rows += count_data_lines(raw)
            tables.append(with_lineage(
                read_csv_as_text(raw),
                _publication_month=month,
                _source_file=zip_path.name,
                _source_member=member,
                _source_sha256=sha,
            ))
    table = pa.concat_tables(tables)
    assert table.num_rows == raw_rows, f"{zip_path.name}: {table.num_rows:,} rows written vs {raw_rows:,} in CSV"
    pq.write_table(table, out_path, compression="zstd")
    return {
        "bronze_file": str(out_path.relative_to(PROJECT_DIR)),
        "rows": table.num_rows,
        "source_columns": "|".join(c for c in table.column_names if not c.startswith("_")),
        "source_file": str(zip_path.relative_to(PROJECT_DIR)),
        "source_sha256": sha,
        "members": len(tables),
    }


sha_by_path = dict(zip(manifest["path"], manifest["sha256"]))
inventory = []
for month_dir in sorted((BRONZE_DIR / "telephony").iterdir()):
    for zip_path in sorted(month_dir.glob("*.zip")):
        kind = next(v for k, v in KINDS.items() if k in zip_path.name)
        sha = sha_by_path[str(zip_path.relative_to(PROJECT_DIR))]
        inventory.append(zip_to_parquet(zip_path, month_dir / f"{kind}.parquet", month_dir.name, sha))
        print(f"{month_dir.name}  {kind:<15} {inventory[-1]['rows']:>10,} rows")

2025-10  day_time         3,714,703 rows
2025-10  durations        2,733,623 rows
2025-11  day_time         3,536,837 rows
2025-11  durations        2,583,806 rows
2025-12  day_time         3,706,340 rows
2025-12  durations        2,689,838 rows
2026-01  day_time         3,811,913 rows
2026-01  durations        2,744,540 rows
2026-02  day_time         3,540,227 rows
2026-02  durations        2,588,234 rows
2026-03  day_time         4,008,498 rows
2026-03  durations        2,923,114 rows
2026-04  day_time         3,836,872 rows
2026-04  durations        2,714,975 rows
2026-05  day_time         3,713,472 rows
2026-05  durations        2,599,947 rows
2026-06  day_time         4,033,904 rows
2026-06  durations        2,925,540 rows
2026-06  calls_answered   1,691,631 rows
2026-07  day_time         4,172,660 rows
2026-07  durations        3,033,331 rows
2026-07  calls_answered   1,755,438 rows
2026-08  day_time         3,798,314 rows
2026-08  durations        2,689,158 rows
2026-08  calls_a

## Step 5 — Has the file layout changed between months?

The October 2025 and November 2025 files are `_v2` reissues, and March 2026 carries an
`_0526` suffix. If NHS England changed a column name or order between months, Silver
needs to know before it stacks the months. This table shows each distinct column list
per file type and which months use it.

In [6]:
inv = pd.DataFrame(inventory)
inv["kind"] = inv["bronze_file"].str.extract(r"/([a-z_]+)\.parquet$")
inv["month"] = inv["bronze_file"].str.extract(r"/(\d{4}-\d{2})/")
schemas = (inv.groupby(["kind", "source_columns"])["month"]
              .agg(lambda m: f"{m.min()} to {m.max()} ({len(m)} months)")
              .reset_index())
pd.set_option("display.max_colwidth", None)
schemas

,kind,source_columns,month
0,calls_answered,Date|REGION_CODE|ICB_CODE|SUB_ICB_LOCATION_CODE|PCN_CODE|PRACTICE_CODE|Indicator|Wait_Time|Core_Category|Value,2026-06 to 2026-08 (3 months)
1,day_time,Date|REGION_CODE|ICB_CODE|SUB_ICB_LOCATION_CODE|PCN_CODE|PRACTICE_CODE|Indicator|Time_Category|Value,2025-10 to 2026-08 (11 months)
2,durations,Date|REGION_CODE|ICB_CODE|SUB_ICB_LOCATION_CODE|PCN_CODE|PRACTICE_CODE|Indicator|Duration|Value,2025-10 to 2026-08 (11 months)


## Step 6 — Registered patients and the ODS practice register

**Registered patients (1 August 2026).** There are two files: practice totals and the
practice-to-PCN/sub-ICB/ICB/region mapping. Each is a single CSV in a ZIP, and they get
the same treatment as the telephony files.

**ODS `epraccur`.** This is the GP practice register, one row per organisation. The file
has **no header row**, and the current ODS export doesn't match the column layout I
expected from older ODS documentation: column 13 reads `ACTIVE` rather than a one-letter
status code, and column 26 holds a role code (`RO76`) where I expected the prescribing
setting. Guessing column names here would be a decision dressed up as a fact, so Bronze
stores them as `col_01` to `col_27`. Silver names them once I've confirmed the layout
against the ODS specification.

In [7]:
for zip_path in sorted((BRONZE_DIR / "registered_patients").rglob("*.zip")):
    sha = sha_by_path[str(zip_path.relative_to(PROJECT_DIR))]
    out = zip_path.with_suffix(".parquet")
    inventory.append(zip_to_parquet(zip_path, out, zip_path.parent.name, sha))
    print(f"{zip_path.name:<28} {inventory[-1]['rows']:>8,} rows")

for csv_path in sorted((BRONZE_DIR / "ods").rglob("epraccur.csv")):
    raw = csv_path.read_bytes()
    n_cols = len(next(iter(pacsv.open_csv(pa.BufferReader(raw),
                 read_options=pacsv.ReadOptions(autogenerate_column_names=True)))).schema)
    names = [f"col_{i:02d}" for i in range(1, n_cols + 1)]
    table = pacsv.read_csv(
        pa.BufferReader(raw),
        read_options=pacsv.ReadOptions(column_names=names),
        convert_options=pacsv.ConvertOptions(column_types={c: pa.string() for c in names},
                                             strings_can_be_null=False),
    )
    assert table.num_rows == count_data_lines(raw) + 1, "epraccur row count mismatch"
    sha = sha_by_path[str(csv_path.relative_to(PROJECT_DIR))]
    table = with_lineage(table, _snapshot=csv_path.parent.name.replace("epraccur_", ""),
                         _source_file=csv_path.name, _source_sha256=sha)
    out = csv_path.with_suffix(".parquet")
    pq.write_table(table, out, compression="zstd")
    inventory.append({"bronze_file": str(out.relative_to(PROJECT_DIR)), "rows": table.num_rows,
                      "source_columns": "|".join(names), "source_file": str(csv_path.relative_to(PROJECT_DIR)),
                      "source_sha256": sha, "members": 1})
    print(f"{csv_path.parent.name:<28} {table.num_rows:>8,} rows x {n_cols} cols")

gp-reg-pat-prac-all.zip         6,129 rows
gp-reg-pat-prac-map.zip         6,129 rows
epraccur_2026-10-02            15,659 rows x 27 cols


## Step 7 — The Excel files stay as they are

The Publication Summary, the participation-by-practice annex and the metadata workbook
are formatted for people to read: title pages, notes rows, merged two-row headers, and
numbers sitting next to their percentages. Turning those into tables means deciding which
rows are headers, and that's a Silver decision. Bronze keeps the `.xlsx` files exactly
as downloaded, fingerprinted in the manifest, and doesn't parse them.

That matters because the published headline figures I have to reconcile against live in
those workbooks (Summary Table 1 for coverage, Table 3 for outcomes, Table 2 for the
national day-and-time grid). Silver parses them, and Gold reconciles against them.

## Step 8 — Bronze inventory

One row for every Parquet file Bronze wrote: where it came from, the source fingerprint,
the row count and the source columns. This is written to `evidence/` (committed), so the
audit trail survives even though the Parquet files themselves are gitignored.

In [8]:
inventory_df = pd.DataFrame(inventory)
inventory_df.insert(0, "built_utc", datetime.now(timezone.utc).isoformat(timespec="seconds"))
inventory_df.insert(1, "manifest", MANIFEST_PATH.name)
inventory_path = EVIDENCE_DIR / "bronze_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print(f"{len(inventory_df)} Bronze files, {inventory_df['rows'].sum():,} rows -> {inventory_path.relative_to(PROJECT_DIR)}")
inventory_df[["bronze_file", "rows", "members"]]

28 Bronze files, 77,095,522 rows -> evidence/bronze_inventory.csv


,bronze_file,rows,members
0,data/bronze/telephony/2025-10/day_time.parquet,3714703,8
1,data/bronze/telephony/2025-10/durations.parquet,2733623,8
2,data/bronze/telephony/2025-11/day_time.parquet,3536837,8
3,data/bronze/telephony/2025-11/durations.parquet,2583806,8
4,data/bronze/telephony/2025-12/day_time.parquet,3706340,8
5,data/bronze/telephony/2025-12/durations.parquet,2689838,8
6,data/bronze/telephony/2026-01/day_time.parquet,3811913,8
7,data/bronze/telephony/2026-01/durations.parquet,2744540,8
8,data/bronze/telephony/2026-02/day_time.parquet,3540227,8
9,data/bronze/telephony/2026-02/durations.parquet,2588234,8


## What's next — Silver

Bronze answers "is this exactly what was published?" Silver answers "what does it mean?"
The Silver jobs already known from profiling:

1. Cast `Value` to a number, and map `CBT001`–`CBT007` to names from the metadata workbook.
2. Hold the `Unassigned` / `Unmapped` rows out of practice-level joins, but keep them in
   national totals.
3. Explain why the practice-level CSVs have 4,990 practice codes against 5,327 included
   practices.
4. Name the `epraccur` columns against the ODS specification.
5. Parse the Summary workbook tables that the reconciliation gate checks against.